# Generative Models

Discriminative models learn a conditional distribution $p(y \mid \mathbf{x})$ — a decision boundary mapping inputs to labels. **Generative models** take the harder task of learning the data distribution $p(\mathbf{x})$ itself, so that new, plausible samples can be drawn from it. In this notebook we study two families that dominated the deep generative modeling literature. The first, **Variational Autoencoders** (VAEs) [@vae], frames generation as approximate Bayesian inference: a learned encoder approximates the posterior $q_\phi(\mathbf{z}|\mathbf{x})$ over a continuous latent space, and a learned decoder approximates the likelihood $p_\theta(\mathbf{x}|\mathbf{z})$. Training maximizes a lower bound on the log-likelihood known as the ELBO. The second, **Generative Adversarial Networks** (GANs) [@gan], takes an adversarial approach: a generator $G$ tries to fool a discriminator $D$ into classifying its outputs as real, while $D$ tries to distinguish generated from real data.

We train both families on MNIST and compare their qualitative properties: VAEs produce a structured, interpolatable latent space at the cost of some blurriness; GANs can produce sharper samples but require careful balancing of two competing objectives. We refer to the [autodiff notebook](../03-autodiff.html) for the basic autoencoder, and to the [CNN notebook](../06-cnn.html) for the convolutional primitives used in the GAN architecture.

<br>

## Preliminaries

Imports and global configuration:

In [ ]:
import math
import torch
import random
import warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib_inline import backend_inline
from torchvision import datasets, transforms

DATASET_DIR = Path("./data").absolute()
DATASET_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

def set_seed(s: int):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

set_seed(RANDOM_SEED)
warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

## Data

**Data.** MNIST consists of 60,000 training and 10,000 test grayscale images of handwritten digits, each of size $28 \times 28$ pixels. We prepare two versions: one normalized to $[0, 1]$ for the VAE, whose reconstruction head uses binary cross-entropy (BCE) loss and a Sigmoid output; and one normalized to $[-1, 1]$ for the GAN, whose generator uses a Tanh output layer. Both use a batch size of $128.$

Loading both variants:

In [ ]:
BATCH_SIZE = 128

# VAE version: pixel values in [0, 1]
transform_vae = transforms.Compose([transforms.ToTensor()])

# GAN version: pixel values in [-1, 1]
transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_vae = datasets.MNIST(DATASET_DIR, train=True,  download=True, transform=transform_vae)
test_vae  = datasets.MNIST(DATASET_DIR, train=False, download=True, transform=transform_vae)
train_gan = datasets.MNIST(DATASET_DIR, train=True,  download=True, transform=transform_gan)

loader_vae_train = torch.utils.data.DataLoader(train_vae, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
loader_vae_test  = torch.utils.data.DataLoader(test_vae,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
loader_gan_train = torch.utils.data.DataLoader(train_gan, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)

print(f"Train batches (VAE): {len(loader_vae_train)} | Test samples: {len(test_vae)}")

## Part 1: Variational Autoencoders

### Autoencoders

A plain **autoencoder** compresses input $\mathbf{x} \in \mathbb{R}^d$ to a latent code $\mathbf{z} = f_\phi(\mathbf{x}) \in \mathbb{R}^k$ (with $k \ll d$) via an encoder, then reconstructs the input with a decoder $\hat{\mathbf{x}} = g_\theta(\mathbf{z})$. Training minimizes reconstruction loss $\|\mathbf{x} - \hat{\mathbf{x}}\|^2.$ The problem: the latent space has no imposed structure — two codes $\mathbf{z}_1, \mathbf{z}_2$ corresponding to meaningful inputs may not interpolate meaningfully through $\mathbf{z}_1 + t(\mathbf{z}_2 - \mathbf{z}_1).$ The encoder simply carves out isolated islands in latent space. See the [autodiff notebook](../03-autodiff.html) for a basic autoencoder implementation.

### The ELBO

A VAE treats $\mathbf{z}$ as a **latent random variable** with prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I}).$ The generative model is $p_\theta(\mathbf{x}, \mathbf{z}) = p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z}),$ and we would like to maximize the marginal log-likelihood of each observed $\mathbf{x}$:

$$
\log p_\theta(\mathbf{x}) = \log \int p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})\, d\mathbf{z}.
$$

This integral is intractable for deep networks. We introduce a variational posterior $q_\phi(\mathbf{z} \mid \mathbf{x})$ and use it to write a lower bound. Starting from Jensen's inequality applied to the concave $\log$ function:

$$
\begin{aligned}
\log p_\theta(\mathbf{x})
&= \log \int p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})\, d\mathbf{z} \\[0.75em]
&= \log \int \frac{p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})}{q_\phi(\mathbf{z} \mid \mathbf{x})} \, q_\phi(\mathbf{z} \mid \mathbf{x})\, d\mathbf{z} \\[0.75em]
&= \log \, \mathbb{E}_{q_\phi(\mathbf{z} \mid \mathbf{x})} \left[ \frac{p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})}{q_\phi(\mathbf{z} \mid \mathbf{x})} \right] \\[0.75em]
&\geq \mathbb{E}_{q_\phi(\mathbf{z} \mid \mathbf{x})} \left[ \log \frac{p_\theta(\mathbf{x} \mid \mathbf{z})\, p(\mathbf{z})}{q_\phi(\mathbf{z} \mid \mathbf{x})} \right] \\[0.75em]
&= \underbrace{\mathbb{E}_{q_\phi} [\log p_\theta(\mathbf{x} \mid \mathbf{z})] - D_\text{KL}(q_\phi(\mathbf{z} \mid \mathbf{x}) \| p(\mathbf{z}))}_{\text{ELBO}(\phi, \theta; \mathbf{x})}.
\end{aligned}
$$

Alternatively, one can derive the same bound by writing $\log p_\theta(\mathbf{x}) = \text{ELBO} + D_\text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p_\theta(\mathbf{z}|\mathbf{x}))$ and noting that the KL divergence is non-negative:

$$
\log p_\theta(\mathbf{x}) = \underbrace{\mathbb{E}_{q_\phi}[\log p_\theta(\mathbf{x}|\mathbf{z})] - D_\text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))}_{\text{ELBO}} + D_\text{KL}(q_\phi(\mathbf{z}|\mathbf{x}) \| p_\theta(\mathbf{z}|\mathbf{x})).
$$

Since $D_\text{KL} \geq 0$ always, $\text{ELBO} \leq \log p_\theta(\mathbf{x}),$ and the bound is tight when $q_\phi(\mathbf{z}|\mathbf{x}) = p_\theta(\mathbf{z}|\mathbf{x}).$ The ELBO decomposes into two interpretable terms: (1) the **reconstruction term** $\mathbb{E}_{q_\phi}[\log p_\theta(\mathbf{x}|\mathbf{z})]$ rewards the decoder for producing inputs that look like $\mathbf{x}$ after sampling $\mathbf{z}$ from the encoder; (2) the **regularization term** $-D_\text{KL}(q_\phi \| p)$ penalizes the encoder for drifting away from the prior, which encourages a smooth, connected latent space that supports generation.

### Reparameterization Trick

We choose $q_\phi(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}_\phi(\mathbf{x}),\, \text{diag}(\boldsymbol{\sigma}^2_\phi(\mathbf{x}))),$ so the encoder outputs the mean vector $\boldsymbol{\mu}$ and log-variance vector $\log \boldsymbol{\sigma}^2.$ During training we need to backpropagate through the sampling operation $\mathbf{z} \sim q_\phi.$ Sampling is not differentiable, but the **reparameterization trick** [@vae] circumvents this: we write

$$
\mathbf{z} = \boldsymbol{\mu}_\phi(\mathbf{x}) + \boldsymbol{\sigma}_\phi(\mathbf{x}) \odot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).
$$

Now $\boldsymbol{\epsilon}$ is the stochastic variable and $\mathbf{z}$ is a deterministic, differentiable function of the parameters $\phi$ and the noise $\boldsymbol{\epsilon},$ so gradients flow back through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ without obstruction.

<br>

**KL closed form.** When $q = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ and $p = \mathcal{N}(\mathbf{0}, \mathbf{I}),$ the KL divergence has a closed form. Recall for two Gaussians:

$$
D_\text{KL}(q \| p)
= \mathbb{E}_q \left[\log q(\mathbf{z}) - \log p(\mathbf{z})\right]
= \mathbb{E}_q \left[-\frac{1}{2}\sum_j (\log\sigma_j^2 + 1) + \frac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2)\right],
$$

where we used $\mathbb{E}_q[z_j^2] = \mu_j^2 + \sigma_j^2$ and $\mathbb{E}_q[\log q] = -\frac{1}{2}\sum_j(\log \sigma_j^2 + 1 + \log 2\pi).$ The $\log 2\pi$ terms cancel, giving:

$$
\boxed{D_\text{KL}(q \| p) = -\frac{1}{2} \sum_{j=1}^d \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right).}
$$

This is non-negative (equals zero iff $\boldsymbol{\mu} = \mathbf{0}$ and $\boldsymbol{\sigma}^2 = \mathbf{1}$) and is cheap to compute.

### Architecture and Loss

We implement the VAE with fully-connected encoder and decoder. The encoder maps a flattened $784$-dimensional image to $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2$ via a shared hidden layer. The decoder maps a sampled $\mathbf{z}$ back to a $784$-dimensional vector passed through Sigmoid, producing pixel probabilities in $[0, 1].$

Defining the VAE model:

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.enc_fc   = nn.Linear(784, 400)
        self.enc_mu   = nn.Linear(400, latent_dim)    # <1>
        self.enc_logv = nn.Linear(400, latent_dim)    # <2>

        # Decoder
        self.dec_fc  = nn.Linear(latent_dim, 400)
        self.dec_out = nn.Linear(400, 784)

    def encode(self, x: torch.Tensor):
        x = x.view(-1, 784)
        h = F.relu(self.enc_fc(x))
        return self.enc_mu(h), self.enc_logv(h)

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * log_var)   # <3>
        eps = torch.randn_like(std)      # <4>
        return mu + std * eps            # <5>

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.dec_fc(z))
        return torch.sigmoid(self.dec_out(h))

    def forward(self, x: torch.Tensor):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon = self.decode(z)
        return recon, mu, log_var

1. The mean head $\boldsymbol{\mu}_\phi(\mathbf{x}) \in \mathbb{R}^k$ — the center of the posterior Gaussian.
2. The log-variance head $\log \boldsymbol{\sigma}^2_\phi(\mathbf{x}) \in \mathbb{R}^k$ — we predict log-variance (not variance) since it is unconstrained and numerically more stable.
3. Recover $\boldsymbol{\sigma}$ as $\exp(\frac{1}{2} \log \boldsymbol{\sigma}^2)$.
4. Sample $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ with the same shape as `std`.
5. Apply $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$ — differentiable with respect to $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2.$

The loss function combines per-batch reconstruction loss and KL divergence. We sum pixel-wise BCE over the batch (matching the sum-reduction KL) and then divide by batch size:

In [ ]:
def vae_loss(
    recon_x: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    beta: float = 1.0,
) -> torch.Tensor:
    recon_loss = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())  # <1>
    return (recon_loss + beta * kl_loss) / x.size(0)                      # <2>

1. Closed-form KL: $-\frac{1}{2}\sum_j (1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2).$ Note `log_var.exp()` $= \sigma_j^2.$
2. Normalize by batch size so the loss magnitude is independent of batch size. The `beta` parameter multiplies the KL term: $\beta = 1$ is the standard VAE [@vae]; $\beta > 1$ encourages more disentangled representations at the cost of reconstruction quality ($\beta$-VAE [@betavae]).

:::{.callout-note}
Using `reduction="sum"` for BCE (not `"mean"`) keeps the reconstruction and KL terms on comparable scales: both sum over the $d = 784$ dimensions per sample. Using `"mean"` for BCE would make the two terms differ by a factor of $784.$

:::

### Training

We train the VAE with Adam for 30 epochs. For visualization, we also train a separate $2$-dimensional latent model:

In [ ]:
#| output: false
LATENT_DIM = 16
VAE_EPOCHS = 30

set_seed(RANDOM_SEED)
vae = VAE(latent_dim=LATENT_DIM).to(device)
optimizer_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)

vae_train_losses = []
for epoch in range(VAE_EPOCHS):
    vae.train()
    total_loss = 0.0
    for x, _ in loader_vae_train:
        x = x.to(device)
        optimizer_vae.zero_grad()
        recon, mu, log_var = vae(x)
        loss = vae_loss(recon, x, mu, log_var)
        loss.backward()
        optimizer_vae.step()
        total_loss += loss.item()
    avg = total_loss / len(loader_vae_train)
    vae_train_losses.append(avg)
    print(f"Epoch {epoch+1:>3}/{VAE_EPOCHS}  ELBO loss: {avg:.2f}")

We also train a 2D VAE for latent space visualization:

In [ ]:
#| output: false
set_seed(RANDOM_SEED)
vae2d = VAE(latent_dim=2).to(device)
optimizer_vae2d = torch.optim.Adam(vae2d.parameters(), lr=1e-3)

for epoch in range(VAE_EPOCHS):
    vae2d.train()
    total_loss = 0.0
    for x, _ in loader_vae_train:
        x = x.to(device)
        optimizer_vae2d.zero_grad()
        recon, mu, log_var = vae2d(x)
        loss = vae_loss(recon, x, mu, log_var)
        loss.backward()
        optimizer_vae2d.step()
        total_loss += loss.item()
    avg = total_loss / len(loader_vae_train)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>3}/{VAE_EPOCHS}  ELBO loss: {avg:.2f}")

Plotting the training loss curve:

In [ ]:
#| code-fold: true
#| label: fig-vae-loss
#| fig-cap: "VAE training loss (negative ELBO per sample) over 30 epochs."
plt.figure(figsize=(6, 4))
plt.plot(range(1, VAE_EPOCHS + 1), vae_train_losses, linewidth=2, color="C0", label="ELBO loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend();

### Latent Space Exploration

We visualize the structure of the latent space in two ways: (1) encoding test images and coloring by digit class to check whether the VAE organizes the space semantically, and (2) linearly interpolating between two codes to verify that the decoder produces smooth, meaningful transitions.

Encoding all test images with the 2D VAE and visualizing the latent space:

In [ ]:
#| code-fold: true
#| label: fig-vae-latent
#| fig-cap: "2D latent space of a VAE trained on MNIST. Each point is the posterior mean $\\boldsymbol{\\mu}_\\phi(\\mathbf{x})$ of a test image, colored by digit class. The prior $\\mathcal{N}(\\mathbf{0}, \\mathbf{I})$ is supported over the central region."
vae2d.eval()
all_mu, all_labels = [], []
with torch.no_grad():
    for x, y in loader_vae_test:
        x = x.to(device)
        mu, _ = vae2d.encode(x)
        all_mu.append(mu.cpu())
        all_labels.append(y)

all_mu     = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    all_mu[:, 0], all_mu[:, 1],
    c=all_labels, cmap="tab10", s=4, alpha=0.6
)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label("Digit class")
ax.set_xlabel(r"$z_1$")
ax.set_ylabel(r"$z_2$")
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

**Figure.** The KL regularization pulls each posterior toward $\mathcal{N}(\mathbf{0}, \mathbf{I}),$ resulting in a compact, relatively smooth latent space. Different digit classes cluster in distinct but overlapping regions, reflecting shared visual structure (e.g. $1$ and $7$ overlap; $4$ and $9$ are adjacent).

Generating new digits by sampling $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decoding:

In [ ]:
#| code-fold: true
#| label: fig-vae-samples
#| fig-cap: "Samples generated by the VAE ($k=16$) by drawing $\\mathbf{z} \\sim \\mathcal{N}(\\mathbf{0}, \\mathbf{I})$ and decoding."
set_seed(RANDOM_SEED)
vae.eval()
nrow, ncol = 5, 10
with torch.no_grad():
    z_sample = torch.randn(nrow * ncol, LATENT_DIM, device=device)
    samples  = vae.decode(z_sample).cpu().view(-1, 28, 28)

fig, axes = plt.subplots(nrow, ncol, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i], cmap="gray")
    ax.axis("off")
fig.tight_layout()
plt.show();

Interpolating between two test images in latent space:

In [ ]:
#| code-fold: true
#| label: fig-vae-interp
#| fig-cap: "Linear interpolation between two test images in the $k=16$ VAE latent space. Each column is a decoding of $\\mathbf{z} = (1-t)\\mathbf{z}_1 + t\\mathbf{z}_2$ for $t \\in [0, 1]$. The top and bottom rows start from different image pairs."
set_seed(RANDOM_SEED)
vae.eval()

n_steps = 10
n_pairs = 4

# Sample 2 * n_pairs images from the test set
test_imgs = [test_vae[i][0] for i in range(2 * n_pairs)]
test_imgs = torch.stack(test_imgs).to(device)

with torch.no_grad():
    mu_all, _ = vae.encode(test_imgs)  # (2*n_pairs, LATENT_DIM)

interp_images = []
for pair in range(n_pairs):
    z1 = mu_all[2 * pair]
    z2 = mu_all[2 * pair + 1]
    alphas = torch.linspace(0, 1, n_steps, device=device)
    with torch.no_grad():
        zs = torch.stack([(1 - a) * z1 + a * z2 for a in alphas])  # (n_steps, k)
        decoded = vae.decode(zs).cpu().view(n_steps, 28, 28)
    interp_images.append(decoded)

fig, axes = plt.subplots(n_pairs, n_steps, figsize=(12, 5))
for row in range(n_pairs):
    for col in range(n_steps):
        axes[row, col].imshow(interp_images[row][col], cmap="gray")
        axes[row, col].axis("off")
fig.tight_layout()
plt.show();

**Figure.** Each row interpolates between two test images. The smooth transitions confirm that the KL regularization produces a continuous latent space: intermediate codes decode to plausible, gradually morphing digits rather than noise.

## Part 2: Generative Adversarial Networks

### The GAN Objective

GANs [@gan] frame generation as a two-player zero-sum game. The **generator** $G\colon \mathbb{R}^{d_z} \to \mathbb{R}^{28 \times 28}$ maps random noise $\mathbf{z} \sim p_\mathbf{z} = \mathcal{N}(\mathbf{0}, \mathbf{I})$ to images; its goal is to produce outputs indistinguishable from real data. The **discriminator** $D\colon \mathbb{R}^{28 \times 28} \to [0,1]$ outputs the probability that its input is a real image; its goal is to correctly classify real vs. generated images. The minimax objective is:

$$
\min_G \max_D \; \mathbb{E}_{\mathbf{x} \sim p_\text{data}}[\log D(\mathbf{x})] + \mathbb{E}_{\mathbf{z} \sim p_\mathbf{z}}[\log(1 - D(G(\mathbf{z})))].
$$

The inner maximization trains $D$ as a binary classifier: real samples labeled $1$, generated samples labeled $0.$ Goodfellow et al. [@gan] show that the optimal discriminator for a fixed generator is $D^*(\mathbf{x}) = p_\text{data}(\mathbf{x}) / (p_\text{data}(\mathbf{x}) + p_G(\mathbf{x})),$ and that the global minimax optimum is achieved when $p_G = p_\text{data},$ at which point $D^* = 1/2$ everywhere and the loss equals $-\log 4.$

<br>

**Non-saturating heuristic.** In practice, minimizing $\mathbb{E}[\log(1 - D(G(\mathbf{z})))]$ w.r.t. $G$ provides weak gradients early in training when $D$ easily distinguishes the generator's poor outputs. The standard fix is the **non-saturating** variant: instead, we maximize $\mathbb{E}[\log D(G(\mathbf{z}))]$ w.r.t. $G,$ which has the same fixed points but provides much stronger gradients when $D(G(\mathbf{z})) \approx 0.$

### DCGAN Architecture

DCGAN [@dcgan] replaces fully-connected layers with strided and transposed convolutions, making the architecture scale to higher-resolution images. The key design choices are: (1) batch normalization in both $G$ and $D,$ except the input layer of $D$ and the output layer of $G$; (2) LeakyReLU (negative slope $0.2$) throughout $D$ to allow gradients to flow through negative activations; (3) ReLU throughout $G$ except the output; (4) Tanh output in $G,$ matching the $[-1,1]$-normalized inputs to $D;$ (5) no pooling layers — downsampling is done by strided convolutions in $D$ and upsampling by transposed convolutions in $G.$

For MNIST ($28 \times 28$) we use a compact three-block architecture. The generator maps a noise vector $\mathbf{z} \in \mathbb{R}^{100}$ treated as a $1 \times 1$ feature map to a $28 \times 28$ image:

$$
\underbrace{(B, 100, 1, 1)}_{\text{noise}} \xrightarrow{\text{ConvT}(7,1,0)} \underbrace{(B, 2f, 7, 7)}_{} \xrightarrow{\text{ConvT}(4,2,1)} \underbrace{(B, f, 14, 14)}_{} \xrightarrow{\text{ConvT}(4,2,1)} \underbrace{(B, 1, 28, 28)}_{\text{image}}
$$

where $f = 64$ is the base feature map count. The discriminator is the spatial mirror.

Defining the generator:

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim: int = 100, ngf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            # (B, z_dim, 1, 1) -> (B, ngf*2, 7, 7)
            nn.ConvTranspose2d(z_dim, ngf * 2, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(inplace=True),
            # (B, ngf*2, 7, 7) -> (B, ngf, 14, 14)
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(inplace=True),
            # (B, ngf, 14, 14) -> (B, 1, 28, 28)
            nn.ConvTranspose2d(ngf, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),  # <1>
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

1. Tanh maps outputs to $[-1, 1],$ matching the normalized GAN training data.

Defining the discriminator:

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ndf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            # (B, 1, 28, 28) -> (B, ndf, 14, 14)  — no BN on first layer
            nn.Conv2d(1, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),  # <1>
            # (B, ndf, 14, 14) -> (B, ndf*2, 7, 7)
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # (B, ndf*2, 7, 7) -> (B, 1, 1, 1)
            nn.Conv2d(ndf * 2, 1, kernel_size=7, stride=1, padding=0, bias=False),
            nn.Sigmoid(),  # <2>
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)  # flatten to (B,)

1. LeakyReLU with slope $0.2$ allows gradients to flow for negative activations, avoiding dead neurons in $D.$
2. Sigmoid maps the scalar logit to a probability in $[0, 1].$

### Training

GAN training alternates between updating $D$ and $G.$ At each step we: (1) update $D$ on a real batch (soft label $0.9$ instead of $1.0$ for label smoothing) and a fake batch (label $0$); (2) update $G$ to maximize $\log D(G(\mathbf{z})),$ i.e. we pass the fake batch through $D$ with label $1$ (non-saturating). Both use Adam with learning rate $2 \times 10^{-4}$ and momentum parameters $\beta_1 = 0.5,$ $\beta_2 = 0.999$ as recommended by [@dcgan].

Training DCGAN for 50 epochs:

In [ ]:
#| output: false
Z_DIM     = 100
NGF       = 64
NDF       = 64
GAN_EPOCHS = 50
REAL_LABEL = 0.9   # label smoothing
FAKE_LABEL = 0.0

set_seed(RANDOM_SEED)
netG = Generator(z_dim=Z_DIM, ngf=NGF).to(device)
netD = Discriminator(ndf=NDF).to(device)

criterion = nn.BCELoss()
optimizer_D = torch.optim.Adam(netD.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_G = torch.optim.Adam(netG.parameters(), lr=2e-4, betas=(0.5, 0.999))

# Fixed noise for visualization
fixed_noise = torch.randn(50, Z_DIM, 1, 1, device=device)

g_losses, d_losses = [], []

for epoch in range(GAN_EPOCHS):
    epoch_g, epoch_d = 0.0, 0.0
    for real_imgs, _ in loader_gan_train:
        B = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # ---- Train Discriminator ----
        netD.zero_grad()
        # Real batch
        label_real = torch.full((B,), REAL_LABEL, device=device)
        d_real = netD(real_imgs)
        loss_d_real = criterion(d_real, label_real)
        loss_d_real.backward()
        # Fake batch
        noise = torch.randn(B, Z_DIM, 1, 1, device=device)
        fake_imgs = netG(noise)
        label_fake = torch.full((B,), FAKE_LABEL, device=device)
        d_fake = netD(fake_imgs.detach())  # <1>
        loss_d_fake = criterion(d_fake, label_fake)
        loss_d_fake.backward()
        optimizer_D.step()
        loss_d = loss_d_real.item() + loss_d_fake.item()

        # ---- Train Generator ----
        netG.zero_grad()
        label_gen = torch.full((B,), REAL_LABEL, device=device)  # <2>
        d_fake2 = netD(fake_imgs)
        loss_g = criterion(d_fake2, label_gen)
        loss_g.backward()
        optimizer_G.step()

        epoch_g += loss_g.item()
        epoch_d += loss_d

    n = len(loader_gan_train)
    g_losses.append(epoch_g / n)
    d_losses.append(epoch_d / n)
    print(f"Epoch {epoch+1:>3}/{GAN_EPOCHS}  G: {epoch_g/n:.4f}  D: {epoch_d/n:.4f}")

1. We call `.detach()` so that gradients from the $D$ update do not propagate into $G.$
2. Non-saturating heuristic: the generator loss uses the real label $1$ on fake images, maximizing $\log D(G(\mathbf{z}))$ instead of minimizing $\log(1 - D(G(\mathbf{z}))).$

:::{.callout-caution}
If $D$ becomes too strong early in training, $G$ receives near-zero gradients and training stalls. Label smoothing (replacing $1$ with $0.9$ for real labels) partially mitigates this by keeping $D$ from becoming overconfident.

:::

Plotting generator and discriminator loss curves:

In [ ]:
#| code-fold: true
#| label: fig-gan-loss
#| fig-cap: "GAN training losses over 50 epochs. Unlike a standard supervised loss, neither curve converges to zero — at equilibrium both should plateau near a balanced value."
epochs_range = range(1, GAN_EPOCHS + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, g_losses, linewidth=2, color="C0", label="Generator loss")
plt.plot(epochs_range, d_losses, linewidth=2, color="C1", label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend();

Displaying the final generated samples from the fixed noise vector:

In [ ]:
#| code-fold: true
#| label: fig-gan-samples
#| fig-cap: "Samples generated by the DCGAN after 50 epochs of training on MNIST."
netG.eval()
with torch.no_grad():
    gan_samples = netG(fixed_noise).cpu().squeeze(1)  # (50, 28, 28)
    # Rescale from [-1, 1] to [0, 1] for display
    gan_samples = (gan_samples + 1.0) / 2.0

fig, axes = plt.subplots(5, 10, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(gan_samples[i], cmap="gray")
    ax.axis("off")
fig.tight_layout()
plt.show();

## Comparison

The two frameworks embody fundamentally different trade-offs:

| Property | VAE | GAN |
| :-- | :-- | :-- |
| **Training stability** | Stable (single loss, closed-form KL) | Can be unstable (two competing objectives) |
| **Sample sharpness** | Tends toward blurriness (Gaussian likelihood averages) | Typically sharper (adversarial pressure) |
| **Latent space** | Structured, interpolatable (KL prior) | Implicit; no encoder |
| **Mode coverage** | Generally good coverage | Prone to mode collapse |
| **Generation** | Sample $\mathbf{z} \sim p(\mathbf{z}),$ decode | Sample $\mathbf{z} \sim p_\mathbf{z},$ pass through $G$ |
| **Likelihood** | Explicit lower bound (ELBO) | No tractable likelihood |

: {tbl-colwidths="[22,39,39]"}

The VAE's blurriness arises because pixel-wise BCE (a proxy for $\log p_\theta(\mathbf{x}|\mathbf{z})$ under a Bernoulli decoder) penalizes each pixel independently, causing the model to average over uncertainty in the decoder rather than commit to sharp features. The GAN, having no explicit reconstruction loss, can produce crisp images but may focus on a small subset of the digit modes.

Side-by-side comparison of 50 VAE and 50 GAN samples:

In [ ]:
#| code-fold: true
#| label: fig-comparison
#| fig-cap: "(**Top**) 50 samples from the VAE ($k=16$). (**Bottom**) 50 samples from the DCGAN. Both are drawn from their respective priors on the same random seed."
set_seed(RANDOM_SEED)
vae.eval()
netG.eval()

with torch.no_grad():
    z_vae = torch.randn(50, LATENT_DIM, device=device)
    vae_imgs = vae.decode(z_vae).cpu().view(50, 28, 28)

    z_gan = torch.randn(50, Z_DIM, 1, 1, device=device)
    gan_imgs = netG(z_gan).cpu().squeeze(1)
    gan_imgs = (gan_imgs + 1.0) / 2.0

fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for i in range(5):
    for j in range(10):
        axes[i, j].imshow(vae_imgs[i * 10 + j], cmap="gray")
        axes[i, j].axis("off")
for i in range(5):
    for j in range(10):
        axes[5 + i, j].imshow(gan_imgs[i * 10 + j], cmap="gray")
        axes[5 + i, j].axis("off")

# Row labels
axes[0, 0].set_ylabel("VAE", fontsize=11, rotation=0, labelpad=30)
axes[5, 0].set_ylabel("GAN", fontsize=11, rotation=0, labelpad=30)

fig.tight_layout()
plt.show();

**Figure.** The VAE samples (top half) are smoother but cover diverse digit shapes. The GAN samples (bottom half) appear crisper but may exhibit mode collapse toward certain digits. The structural difference reflects the underlying training objectives: the VAE optimizes a reconstruction term that encourages fidelity at the pixel level, while the GAN optimizes an adversarial signal that rewards perceptual plausibility without explicit pixel-level fidelity.

:::{.callout-note}
Both VAEs and GANs have largely been superseded by **diffusion models** for image generation, which achieve higher fidelity and diversity through a learned denoising process. However, the ELBO framework underpins many modern latent variable models, and the adversarial training principle reappears in contexts ranging from domain adaptation to reward modeling.

:::

■